# 02 — Preprocessing Pipeline

This notebook prepares the dataset for modeling.  
The preprocessing steps include:

- Dropping identifier columns not useful for prediction  
- Separating numerical, categorical, and binary feature groups  
- Converting binary (Yes/No) fields into numeric (0/1)  
- Applying one-hot encoding to categorical variables  
- Producing the final cleaned and encoded dataset ready for train-test split and model training  

This ensures all features are in numerical format and suitable for SVMs, Neural Networks, and other machine learning models.

### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
pd.set_option("display.max_columns", None)

### Load Raw Train & Test Data

In [2]:
train = pd.read_csv("../data/raw/train_updated.csv")
test = pd.read_csv("../data/raw/test_updated.csv")

train.head(), test.head()


(    ProfileID  ApplicantYears  AnnualEarnings  RequestedSum  TrustMetric  \
 0  DRIRC89L0T              18          137576        209136          846   
 1  TS0FIUNHNU              47           57194          5970          748   
 2  I0YR284A1V              26           84328         95065          453   
 3  WB1T7NQV8A              53           49795        229582          533   
 4  J6GU9M4G1Z              49          115450         22072          840   
 
    WorkDuration  ActiveAccounts  OfferRate  RepayPeriod  DebtFactor  \
 0            26               2      10.47           60        0.81   
 1            30               2      19.72           36        0.73   
 2             7               2      24.25           12        0.45   
 3           107               3      14.44           60        0.17   
 4             0               4      24.48           12        0.11   
 
   QualificationLevel   WorkCategory RelationshipStatus OwnsProperty  \
 0        High School  Self-em

### Drop ID Column

In [3]:
# Save IDs before ANY transformations
train_ids = train["ProfileID"].copy()
test_ids  = test["ProfileID"].copy()

# Drop the ID column from feature set
train = train.drop("ProfileID", axis=1)
test  = test.drop("ProfileID", axis=1)

### Identify columns as Numerical or Categorical or Binary

In [4]:
# Numeric features
numerical = [
    "ApplicantYears", "AnnualEarnings", "RequestedSum", "TrustMetric",
    "WorkDuration", "ActiveAccounts", "OfferRate", "RepayPeriod", "DebtFactor"
]

# Categorical features
categorical = [
    "QualificationLevel",
    "WorkCategory",
    "RelationshipStatus",
    "FundUseCase"
]

# Binary Yes/No features
binary_cols = ["OwnsProperty", "FamilyObligation", "JointApplicant"]

### Convert BInary Yes/No to 0/1

In [5]:
binary_map = {"Yes": 1, "No": 0}

for col in binary_cols:
    train[col] = train[col].map(binary_map)
    test[col] = test[col].map(binary_map)

### One-Hot Encode Categorical Columns (Train + Test Together)

In [6]:
# One-hot encode train
train_ohe = pd.get_dummies(train, columns=categorical, drop_first=True)

# One-hot encode test
test_ohe = pd.get_dummies(test, columns=categorical, drop_first=True)

# Align test columns to match train
test_ohe = test_ohe.reindex(columns=train_ohe.columns[:-1], fill_value=0)
# (Exclude RiskFlag which is last column in train)


### Extract Target & Features

In [7]:
y = train_ohe["RiskFlag"].copy()
X = train_ohe.drop("RiskFlag", axis=1)

# Test has no RiskFlag
X_test_final = test_ohe.copy()

### Save Processed Files

In [8]:
os.makedirs("../data/processed/processed_full", exist_ok=True)

# Save train
processed_train = X.copy()
processed_train["RiskFlag"] = y
processed_train.to_csv("../data/processed/processed_full/processed_train.csv", index=False)

# Save test
processed_test = X_test_final.copy()
processed_test.to_csv("../data/processed/processed_full/processed_test.csv", index=False)

# Save ProfileID for submissions
train_ids.to_csv("../data/processed/processed_full/train_ids.csv", index=False)
test_ids.to_csv("../data/processed/processed_full/test_ids.csv", index=False)

print("Preprocessing complete and saved.")

Preprocessing complete and saved.
